# E-Commerce Analytics Warehouse: Auditing and Cleaning the Olist Data

In [1]:
import pandas as pd
import os

# This notebook lives in /notebooks, so the data folder is one level up
RAW_DIR = "../data/raw"

print("Running from:", os.getcwd())
print(os.listdir(RAW_DIR))

Running from: C:\Users\ASUS\ecommerce-analytics-warehouse\notebooks
['olist_customers_dataset.csv', 'olist_geolocation_dataset.csv', 'olist_orders_dataset.csv', 'olist_order_items_dataset.csv', 'olist_order_payments_dataset.csv', 'olist_order_reviews_dataset.csv', 'olist_products_dataset.csv', 'olist_sellers_dataset.csv', 'product_category_name_translation.csv']


## Loading the Core Tables and Checking Their Size

In [2]:
customers = pd.read_csv(f"{RAW_DIR}/olist_customers_dataset.csv")
orders    = pd.read_csv(f"{RAW_DIR}/olist_orders_dataset.csv")
items     = pd.read_csv(f"{RAW_DIR}/olist_order_items_dataset.csv")
payments  = pd.read_csv(f"{RAW_DIR}/olist_order_payments_dataset.csv")
reviews   = pd.read_csv(f"{RAW_DIR}/olist_order_reviews_dataset.csv")
products  = pd.read_csv(f"{RAW_DIR}/olist_products_dataset.csv")
sellers   = pd.read_csv(f"{RAW_DIR}/olist_sellers_dataset.csv")
category_translation = pd.read_csv(f"{RAW_DIR}/product_category_name_translation.csv")

In [3]:
print("customers:", customers.shape)
print("orders:", orders.shape)
print("items:", items.shape)
print("payments:", payments.shape)
print("reviews:", reviews.shape)
print("products:", products.shape)
print("sellers:", sellers.shape)
print("category_translation:", category_translation.shape)

customers: (99441, 5)
orders: (99441, 8)
items: (112650, 7)
payments: (103886, 5)
reviews: (99224, 7)
products: (32951, 9)
sellers: (3095, 4)
category_translation: (71, 2)


## Exploring the Orders Table

In [4]:
orders.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [5]:
orders.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   order_id                       99441 non-null  object
 1   customer_id                    99441 non-null  object
 2   order_status                   99441 non-null  object
 3   order_purchase_timestamp       99441 non-null  object
 4   order_approved_at              99281 non-null  object
 5   order_delivered_carrier_date   97658 non-null  object
 6   order_delivered_customer_date  96476 non-null  object
 7   order_estimated_delivery_date  99441 non-null  object
dtypes: object(8)
memory usage: 6.1+ MB


In [6]:
orders.isnull().sum()

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

In [7]:
orders["order_status"].value_counts()

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

## Converting Order Dates from Text to Real Dates

In [8]:
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]

for col in date_columns:
    orders[col] = pd.to_datetime(orders[col])

In [9]:
print(orders.dtypes)
print()
print("First order:", orders["order_purchase_timestamp"].min())
print("Last order:", orders["order_purchase_timestamp"].max())

order_id                                 object
customer_id                              object
order_status                             object
order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
dtype: object

First order: 2016-09-04 21:15:19
Last order: 2018-10-17 17:30:18


## Exploring the Customers Table

In [10]:
customers.head()

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


In [11]:
customers.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   customer_id               99441 non-null  object
 1   customer_unique_id        99441 non-null  object
 2   customer_zip_code_prefix  99441 non-null  int64 
 3   customer_city             99441 non-null  object
 4   customer_state            99441 non-null  object
dtypes: int64(1), object(4)
memory usage: 3.8+ MB


In [12]:
print("Rows:", len(customers))
print("Unique customer_id:", customers["customer_id"].nunique())
print("Unique customer_unique_id:", customers["customer_unique_id"].nunique())
print("Unique states:", customers["customer_state"].nunique())
print("Unique cities:", customers["customer_city"].nunique())

Rows: 99441
Unique customer_id: 99441
Unique customer_unique_id: 96096
Unique states: 27
Unique cities: 4119


In [13]:
customers["customer_state"].value_counts().head(10)

customer_state
SP    41746
RJ    12852
MG    11635
RS     5466
PR     5045
SC     3637
BA     3380
DF     2140
ES     2033
GO     2020
Name: count, dtype: int64

## Exploring the Products Table

In [14]:
products.head()

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0


In [15]:
products.isnull().sum()

product_id                      0
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64

In [16]:
products["product_category_name"].value_counts().head(10)

product_category_name
cama_mesa_banho           3029
esporte_lazer             2867
moveis_decoracao          2657
beleza_saude              2444
utilidades_domesticas     2335
automotivo                1900
informatica_acessorios    1639
brinquedos                1411
relogios_presentes        1329
telefonia                 1134
Name: count, dtype: int64

In [17]:
category_translation.head(10)

,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto
3,cama_mesa_banho,bed_bath_table
4,moveis_decoracao,furniture_decor
5,esporte_lazer,sports_leisure
6,perfumaria,perfumery
7,utilidades_domesticas,housewares
8,telefonia,telephony
9,relogios_presentes,watches_gifts


## Checking That Every Product Category Has an English Translation

In [18]:
# All the distinct Portuguese category names used in the products table
product_categories = set(products["product_category_name"].dropna().unique())

# All the Portuguese names that the translation table can translate
translated_categories = set(category_translation["product_category_name"])

# Categories used by products but missing from the translation table
missing_translation = product_categories - translated_categories
print(missing_translation)

{'portateis_cozinha_e_preparadores_de_alimentos', 'pc_gamer'}


## Adding the Two Missing Category Translations

In [19]:
missing_rows = pd.DataFrame({
    "product_category_name": [
        "pc_gamer",
        "portateis_cozinha_e_preparadores_de_alimentos",
    ],
    "product_category_name_english": [
        "pc_gamer",
        "portable_kitchen_food_preparers",
    ],
})

category_translation = pd.concat([category_translation, missing_rows], ignore_index=True)

print(category_translation.shape)

(73, 2)


In [20]:
translated_categories = set(category_translation["product_category_name"])
print(product_categories - translated_categories)

set()


## Exploring the Order Items Table

In [21]:
items.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


In [22]:
items.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   order_id             112650 non-null  object 
 1   order_item_id        112650 non-null  int64  
 2   product_id           112650 non-null  object 
 3   seller_id            112650 non-null  object 
 4   shipping_limit_date  112650 non-null  object 
 5   price                112650 non-null  float64
 6   freight_value        112650 non-null  float64
dtypes: float64(2), int64(1), object(4)
memory usage: 6.0+ MB


In [23]:
items[["price", "freight_value"]].describe()

,price,freight_value
count,112650.000000,112650.000000
mean,120.653739,19.990320
std,183.633928,15.806405
min,0.850000,0.000000
25%,39.900000,13.080000
50%,74.990000,16.260000
75%,134.900000,21.150000
max,6735.000000,409.680000


In [24]:
print("Rows:", len(items))
print("Unique orders:", items["order_id"].nunique())
print("Unique products:", items["product_id"].nunique())
print("Unique sellers:", items["seller_id"].nunique())
print()
print("Duplicate (order_id, order_item_id) rows:",
      items.duplicated(subset=["order_id", "order_item_id"]).sum())

Rows: 112650
Unique orders: 98666
Unique products: 32951
Unique sellers: 3095

Duplicate (order_id, order_item_id) rows: 0


## Checking Order Items Against the Orders Table

In [25]:
items["shipping_limit_date"] = pd.to_datetime(items["shipping_limit_date"])
print(items["shipping_limit_date"].dtype)

datetime64[ns]


In [26]:
orders_without_items = orders[~orders["order_id"].isin(items["order_id"])]

print("Orders with no items:", len(orders_without_items))
orders_without_items["order_status"].value_counts()

Orders with no items: 775


order_status
unavailable    603
canceled       164
created          5
invoiced         2
shipped          1
Name: count, dtype: int64

In [27]:
items_without_orders = items[~items["order_id"].isin(orders["order_id"])]
print("Items with no matching order:", len(items_without_orders))

Items with no matching order: 0


## Exploring the Payments Table

In [28]:
payments.head()

,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45


In [29]:
payments.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 103886 entries, 0 to 103885
Data columns (total 5 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   order_id              103886 non-null  object 
 1   payment_sequential    103886 non-null  int64  
 2   payment_type          103886 non-null  object 
 3   payment_installments  103886 non-null  int64  
 4   payment_value         103886 non-null  float64
dtypes: float64(1), int64(2), object(2)
memory usage: 4.0+ MB


In [30]:
payments[["payment_installments", "payment_value"]].describe()

,payment_installments,payment_value
count,103886.000000,103886.000000
mean,2.853349,154.100380
std,2.687051,217.494064
min,0.000000,0.000000
25%,1.000000,56.790000
50%,1.000000,100.000000
75%,4.000000,171.837500
max,24.000000,13664.080000


In [31]:
payments["payment_type"].value_counts()

payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64

In [32]:
print("Rows:", len(payments))
print("Unique orders:", payments["order_id"].nunique())

payments.groupby("order_id").size().value_counts().sort_index().head(10)

Rows: 103886
Unique orders: 99440


1     96479
2      2382
3       301
4       108
5        52
6        36
7        28
8        11
9         9
10        5
Name: count, dtype: int64

## Summarizing Payments to One Row Per Order

In [33]:
payment_totals = payments.groupby("order_id").agg(
    total_payment_value=("payment_value", "sum"),
    payment_methods_used=("payment_type", "nunique"),
    max_installments=("payment_installments", "max"),
).reset_index()

payment_totals.head()

,order_id,total_payment_value,payment_methods_used,max_installments
0,00010242fe8c5a6d1ba2dd792cb16214,72.19,1,2
1,00018f77f2f0320c557190d7a144bdd3,259.83,1,3
2,000229ec398224ef6ca0657da4fc703e,216.87,1,5
3,00024acbcdf0a6daa1e931b038114c75,25.78,1,2
4,00042b26cf59d7ce69dfabb4e55b4fd9,218.04,1,3


In [34]:
# Sort so the biggest payment of each order comes first, then keep that one row
main_payment = (
    payments.sort_values("payment_value", ascending=False)
            .drop_duplicates(subset="order_id", keep="first")
            [["order_id", "payment_type"]]
            .rename(columns={"payment_type": "main_payment_type"})
)

main_payment["main_payment_type"] = main_payment["main_payment_type"].replace("not_defined", "unknown")

main_payment.head()

,order_id,main_payment_type
52107,03caa2c082116e1d31e67e9ae3700499,credit_card
34370,736e1922ae60d0d6a89247b851902527,boleto
41419,0812eb902a67711a1cb742b3cdaa65ae,credit_card
49581,fefacc66af859508bf1a7934eab1e97f,boleto
85539,f5136e38d1a14a4dbd87dff67da82701,boleto


In [35]:
order_payments = payment_totals.merge(main_payment, on="order_id", how="left")

print("Rows:", len(order_payments))
print("Unique orders:", order_payments["order_id"].nunique())
order_payments["main_payment_type"].value_counts()

Rows: 99440
Unique orders: 99440


main_payment_type
credit_card    74975
boleto         19784
voucher         3151
debit_card      1527
unknown            3
Name: count, dtype: int64

## Exploring the Reviews Table

In [36]:
reviews.head()

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18 00:00:00,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10 00:00:00,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17 00:00:00,2018-02-18 14:36:24
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,NaN,Recebi bem antes do prazo estipulado.,2017-04-21 00:00:00,2017-04-21 22:02:06
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,NaN,Parabéns lojas lannister adorei comprar pela I...,2018-03-01 00:00:00,2018-03-02 10:26:53


In [37]:
reviews.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99224 entries, 0 to 99223
Data columns (total 7 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   review_id                99224 non-null  object
 1   order_id                 99224 non-null  object
 2   review_score             99224 non-null  int64 
 3   review_comment_title     11568 non-null  object
 4   review_comment_message   40977 non-null  object
 5   review_creation_date     99224 non-null  object
 6   review_answer_timestamp  99224 non-null  object
dtypes: int64(1), object(6)
memory usage: 5.3+ MB


In [38]:
reviews["review_score"].value_counts().sort_index()

review_score
1    11424
2     3151
3     8179
4    19142
5    57328
Name: count, dtype: int64

In [39]:
print("Rows:", len(reviews))
print("Unique orders:", reviews["order_id"].nunique())
print("Unique review_id:", reviews["review_id"].nunique())
print()
reviews.groupby("order_id").size().value_counts().sort_index()

Rows: 99224
Unique orders: 98673
Unique review_id: 98410



1    98126
2      543
3        4
Name: count, dtype: int64

## Reducing Reviews to One Row Per Order

In [40]:
reviews["review_creation_date"] = pd.to_datetime(reviews["review_creation_date"])
reviews["review_answer_timestamp"] = pd.to_datetime(reviews["review_answer_timestamp"])

print(reviews[["review_creation_date", "review_answer_timestamp"]].dtypes)

review_creation_date       datetime64[ns]
review_answer_timestamp    datetime64[ns]
dtype: object


In [41]:
order_reviews = (
    reviews.sort_values("review_answer_timestamp", ascending=False)
           .drop_duplicates(subset="order_id", keep="first")
           .copy()
)

# Yes/no flag: did the customer write a comment?
order_reviews["has_comment"] = order_reviews["review_comment_message"].notnull()

# Keep only the columns we need
order_reviews = order_reviews[["order_id", "review_score", "review_creation_date",
                               "review_answer_timestamp", "has_comment"]]

print("Rows:", len(order_reviews))
print("Unique orders:", order_reviews["order_id"].nunique())
order_reviews.head()

Rows: 98673
Unique orders: 98673


,order_id,review_score,review_creation_date,review_answer_timestamp,has_comment
80582,30a2f24dd6770c91faa6b3481319204b,5,2018-08-14,2018-10-29 12:27:35,False
49615,7e8072dc0f35ebb0c1b2a4743e0f179a,5,2018-07-12,2018-10-26 21:36:41,False
92802,7ce4e38f4eadd993bb5b2e60bb7f7bec,5,2018-08-16,2018-10-24 21:34:38,True
14687,99e4a1f9377bf28f08f54a2eecccbf0d,5,2018-02-02,2018-10-24 18:26:25,False
46904,0b223d92c27432930dfe407c6aea3041,5,2018-08-31,2018-10-24 16:27:36,False


In [42]:
print("Average review score:", round(order_reviews["review_score"].mean(), 2))
print("Share of reviews with a comment:", round(order_reviews["has_comment"].mean() * 100, 1), "%")
order_reviews["review_score"].value_counts().sort_index()

Average review score: 4.09
Share of reviews with a comment: 41.3 %


review_score
1    11363
2     3131
3     8133
4    19038
5    57008
Name: count, dtype: int64

## Exploring the Sellers Table

In [43]:
sellers.head()

,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ
3,c0f3eea2e14555b6faeea3dd58c1b1c3,4195,sao paulo,SP
4,51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP


In [44]:
print("Rows:", len(sellers))
print("Unique sellers:", sellers["seller_id"].nunique())
print("Missing values:", sellers.isnull().sum().sum())
print()
sellers["seller_state"].value_counts().head(5)

Rows: 3095
Unique sellers: 3095
Missing values: 0



seller_state
SP    1849
PR     349
MG     244
SC     190
RJ     171
Name: count, dtype: int64

## Exploring the Geolocation Table

In [45]:
geolocation = pd.read_csv(
    f"{RAW_DIR}/olist_geolocation_dataset.csv",
    dtype={"geolocation_zip_code_prefix": str},
)

print(geolocation.shape)
geolocation.head()

(1000163, 5)


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,01037,-23.545621,-46.639292,sao paulo,SP
1,01046,-23.546081,-46.644820,sao paulo,SP
2,01046,-23.546129,-46.642951,sao paulo,SP
3,01041,-23.544392,-46.639499,sao paulo,SP
4,01035,-23.541578,-46.641607,sao paulo,SP


In [46]:
print("Rows:", len(geolocation))
print("Unique zip code prefixes:", geolocation["geolocation_zip_code_prefix"].nunique())
print("Fully duplicated rows:", geolocation.duplicated().sum())
print()
geolocation[["geolocation_lat", "geolocation_lng"]].describe()

Rows: 1000163
Unique zip code prefixes: 19015
Fully duplicated rows: 261831



,geolocation_lat,geolocation_lng
count,1.000163e+06,1.000163e+06
mean,-2.117615e+01,-4.639054e+01
std,5.715866e+00,4.269748e+00
min,-3.660537e+01,-1.014668e+02
25%,-2.360355e+01,-4.857317e+01
50%,-2.291938e+01,-4.663788e+01
75%,-1.997962e+01,-4.376771e+01
max,4.506593e+01,1.211054e+02


## Cleaning Zip Codes and Building a One-Row-Per-Zip Location Table


In [47]:
customers["customer_zip_code_prefix"] = (
    customers["customer_zip_code_prefix"].astype(str).str.zfill(5)
)
sellers["seller_zip_code_prefix"] = (
    sellers["seller_zip_code_prefix"].astype(str).str.zfill(5)
)

customers[["customer_zip_code_prefix"]].head()

,customer_zip_code_prefix
0,14409
1,09790
2,01151
3,08775
4,13056


In [48]:
inside_brazil = (
    geolocation["geolocation_lat"].between(-34, 5.3)
    & geolocation["geolocation_lng"].between(-74, -34)
)

print("Points outside Brazil:", (~inside_brazil).sum())

geolocation_clean = geolocation[inside_brazil].copy()
print("Rows kept:", len(geolocation_clean))

Points outside Brazil: 42
Rows kept: 1000121


In [49]:
zip_locations = (
    geolocation_clean.groupby("geolocation_zip_code_prefix")
    .agg(
        latitude=("geolocation_lat", "median"),
        longitude=("geolocation_lng", "median"),
    )
    .reset_index()
    .rename(columns={"geolocation_zip_code_prefix": "zip_code_prefix"})
)

print("Rows:", len(zip_locations))
zip_locations.head()

Rows: 19010


,zip_code_prefix,latitude,longitude
0,01001,-23.550381,-46.634027
1,01002,-23.548551,-46.635072
2,01003,-23.548977,-46.635313
3,01004,-23.549535,-46.634771
4,01005,-23.549612,-46.636532


In [50]:
known_zips = set(zip_locations["zip_code_prefix"])

customer_match = customers["customer_zip_code_prefix"].isin(known_zips).mean() * 100
seller_match = sellers["seller_zip_code_prefix"].isin(known_zips).mean() * 100

print(f"Customers with a known location: {customer_match:.2f}%")
print(f"Sellers with a known location: {seller_match:.2f}%")

Customers with a known location: 99.72%
Sellers with a known location: 99.77%


## Cleaning the Products Table

In [51]:
products = products.rename(columns={
    "product_name_lenght": "product_name_length",
    "product_description_lenght": "product_description_length",
})

products.columns.tolist()

['product_id',
 'product_category_name',
 'product_name_length',
 'product_description_length',
 'product_photos_qty',
 'product_weight_g',
 'product_length_cm',
 'product_height_cm',
 'product_width_cm']

In [52]:
products_clean = products.merge(category_translation, on="product_category_name", how="left")

products_clean["product_category_english"] = (
    products_clean["product_category_name_english"].fillna("unknown")
)

products_clean = products_clean.drop(columns=["product_category_name_english"])

print("Rows:", len(products_clean))
products_clean[["product_category_name", "product_category_english"]].head()

Rows: 32951


,product_category_name,product_category_english
0,perfumaria,perfumery
1,artes,art
2,esporte_lazer,sports_leisure
3,bebes,baby
4,utilidades_domesticas,housewares


In [53]:
print("Unknown category:", (products_clean["product_category_english"] == "unknown").sum())
print("Different categories:", products_clean["product_category_english"].nunique())
print()
products_clean["product_category_english"].value_counts().head(10)

Unknown category: 610
Different categories: 74



product_category_english
bed_bath_table           3029
sports_leisure           2867
furniture_decor          2657
health_beauty            2444
housewares               2335
auto                     1900
computers_accessories    1639
toys                     1411
watches_gifts            1329
telephony                1134
Name: count, dtype: int64

## Cleaning the Orders Table and Creating Delivery Metrics

In [54]:
orders_clean = orders.copy()

# Which orders count for delivery metrics: delivered AND with a delivery date
delivered_mask = (
    (orders_clean["order_status"] == "delivered")
    & orders_clean["order_delivered_customer_date"].notnull()
)

# Purchase date without the time of day
orders_clean["purchase_date"] = orders_clean["order_purchase_timestamp"].dt.normalize()

# Days from purchase to delivery
orders_clean["delivery_days"] = (
    (orders_clean["order_delivered_customer_date"] - orders_clean["order_purchase_timestamp"])
    .dt.total_seconds() / 86400
).round(1).where(delivered_mask)

# Days between the promised date and the actual delivery (positive = late)
orders_clean["delay_days"] = (
    orders_clean["order_delivered_customer_date"].dt.normalize()
    - orders_clean["order_estimated_delivery_date"].dt.normalize()
).dt.days.where(delivered_mask)

# True when delivered after the promised date
orders_clean["is_late"] = orders_clean["delay_days"] > 0

orders_clean[["order_id", "order_status", "delivery_days", "delay_days", "is_late"]].head()

,order_id,order_status,delivery_days,delay_days,is_late
0,e481f51cbdc54678b7cc49136f2d6af7,delivered,8.4,-8.0,False
1,53cdb2fc8bc7dce0b6741e2150273451,delivered,13.8,-6.0,False
2,47770eb9100c2d0c44946d9cf07ec65d,delivered,9.4,-18.0,False
3,949d5b44dbf5de918fe9c16f97b45f8a,delivered,13.2,-13.0,False
4,ad21c59c0840e6cb83a9ceb5573f8159,delivered,2.9,-10.0,False


In [55]:
measured = orders_clean[delivered_mask]

print("Orders with delivery metrics:", len(measured))
print("Average delivery days:", round(measured["delivery_days"].mean(), 1))
print("Median delivery days:", round(measured["delivery_days"].median(), 1))
print("Late deliveries:", f"{measured['is_late'].mean() * 100:.1f}%")

Orders with delivery metrics: 96470
Average delivery days: 12.6
Median delivery days: 10.2
Late deliveries: 6.8%


## Building the Customer Dimension

In [56]:
# Attach each customer_id's purchase time, so we can find each person's latest order
customer_orders = customers.merge(
    orders_clean[["customer_id", "order_purchase_timestamp"]],
    on="customer_id",
    how="left",
)

# Keep one row per real person: the one from their most recent order
latest_customer = (
    customer_orders.sort_values("order_purchase_timestamp", ascending=False)
                   .drop_duplicates(subset="customer_unique_id", keep="first")
)

# Add latitude and longitude from the zip code location table
dim_customers = latest_customer.merge(
    zip_locations,
    left_on="customer_zip_code_prefix",
    right_on="zip_code_prefix",
    how="left",
)

# Keep the columns we need, with clean names
dim_customers = dim_customers[[
    "customer_unique_id", "customer_city", "customer_state",
    "customer_zip_code_prefix", "latitude", "longitude",
]].copy()

dim_customers["customer_city"] = dim_customers["customer_city"].str.title()

# Create a simple whole-number key for each customer
dim_customers = dim_customers.reset_index(drop=True)
dim_customers.insert(0, "customer_key", dim_customers.index + 1)

print("Rows:", len(dim_customers))
dim_customers.head()

Rows: 96096


,customer_key,customer_unique_id,customer_city,customer_state,customer_zip_code_prefix,latitude,longitude
0,1,87ab9fec999db8bd5774917de3cdf01c,Sorocaba,SP,18077,-23.454244,-47.475351
1,2,262e1f1e26e92e86375f86840b4ffd63,Picos,PI,64600,-7.085318,-41.470490
2,3,af5454198a97379394cacf676e1e96cb,Registro,SP,11900,-24.498159,-47.844494
3,4,634420a0ea42302205032ed44ac7fccc,Pirai,RJ,27175,-22.626315,-43.901412
4,5,9bb92bebd4cb7511e1a02d5e50bc4655,Guarulhos,SP,07176,-23.409686,-46.388689


In [57]:
print("Unique customer keys:", dim_customers["customer_key"].nunique())
print("Unique customer_unique_id:", dim_customers["customer_unique_id"].nunique())
print("Missing latitude:", dim_customers["latitude"].isnull().sum())
print("Missing state:", dim_customers["customer_state"].isnull().sum())

Unique customer keys: 96096
Unique customer_unique_id: 96096
Missing latitude: 269
Missing state: 0


## Building the Product and Seller Dimensions

In [58]:
dim_products = products_clean[[
    "product_id", "product_category_name", "product_category_english",
    "product_photos_qty", "product_weight_g",
    "product_length_cm", "product_height_cm", "product_width_cm",
]].copy()

# A readable category label for dashboards: "bed_bath_table" becomes "Bed Bath Table"
dim_products["product_category"] = (
    dim_products["product_category_english"].str.replace("_", " ").str.title()
)

# Package volume in cubic centimetres
dim_products["product_volume_cm3"] = (
    dim_products["product_length_cm"]
    * dim_products["product_height_cm"]
    * dim_products["product_width_cm"]
)

# Whole-number key for each product
dim_products = dim_products.reset_index(drop=True)
dim_products.insert(0, "product_key", dim_products.index + 1)

print("Rows:", len(dim_products))
dim_products[["product_key", "product_id", "product_category", "product_weight_g", "product_volume_cm3"]].head()

Rows: 32951


,product_key,product_id,product_category,product_weight_g,product_volume_cm3
0,1,1e9e8ef04dbcff4541ed26657ea517e5,Perfumery,225.0,2240.0
1,2,3aa071139cb16b67ca9e5dea641aaa2f,Art,1000.0,10800.0
2,3,96bd76ec8810374ed1b65e291975717f,Sports Leisure,154.0,2430.0
3,4,cef67bcfe19066a932b7673e239eb23d,Baby,371.0,2704.0
4,5,9dc1a7de274444849c219cff195d0b71,Housewares,625.0,4420.0


In [59]:
dim_sellers = sellers.merge(
    zip_locations,
    left_on="seller_zip_code_prefix",
    right_on="zip_code_prefix",
    how="left",
)

dim_sellers = dim_sellers[[
    "seller_id", "seller_city", "seller_state",
    "seller_zip_code_prefix", "latitude", "longitude",
]].copy()

dim_sellers["seller_city"] = dim_sellers["seller_city"].str.title()

dim_sellers = dim_sellers.reset_index(drop=True)
dim_sellers.insert(0, "seller_key", dim_sellers.index + 1)

print("Rows:", len(dim_sellers))
dim_sellers.head()

Rows: 3095


,seller_key,seller_id,seller_city,seller_state,seller_zip_code_prefix,latitude,longitude
0,1,3442f8959a84dea7ee197c632cb2df15,Campinas,SP,13023,-22.894561,-47.062380
1,2,d1b65fc7debc3361ea86b5f14c68d2e2,Mogi Guacu,SP,13844,-22.382440,-46.947605
2,3,ce3ad9de960102d0677a81f5d0bb7b2d,Rio De Janeiro,RJ,20031,-22.909624,-43.176774
3,4,c0f3eea2e14555b6faeea3dd58c1b1c3,Sao Paulo,SP,04195,-23.657453,-46.612406
4,5,51a04a8a6bdcb23deccc82b0b80742cf,Braganca Paulista,SP,12914,-22.964436,-46.535090


In [60]:
print("Products:", len(dim_products), "| unique keys:", dim_products["product_key"].nunique())
print("Sellers:", len(dim_sellers), "| unique keys:", dim_sellers["seller_key"].nunique())
print("Sellers missing latitude:", dim_sellers["latitude"].isnull().sum())
print()
dim_products["product_category"].value_counts().head(5)

Products: 32951 | unique keys: 32951
Sellers: 3095 | unique keys: 3095
Sellers missing latitude: 7



product_category
Bed Bath Table     3029
Sports Leisure     2867
Furniture Decor    2657
Health Beauty      2444
Housewares         2335
Name: count, dtype: int64

In [61]:
small_words = {" De ": " de ", " Do ": " do ", " Da ": " da ", " Dos ": " dos ", " Das ": " das "}

for old, new in small_words.items():
    dim_customers["customer_city"] = dim_customers["customer_city"].str.replace(old, new, regex=False)
    dim_sellers["seller_city"] = dim_sellers["seller_city"].str.replace(old, new, regex=False)

dim_sellers.loc[dim_sellers["seller_city"].str.contains(" de ", na=False), "seller_city"].head()

2          Rio de Janeiro
5          Rio de Janeiro
15       Lauro de Freitas
24           Tres de Maio
60    Santana de Parnaiba
Name: seller_city, dtype: object

## Building the Date Dimension

In [62]:
# One row for every calendar day from 2016 to 2018
dim_date = pd.DataFrame({"date": pd.date_range("2016-01-01", "2018-12-31", freq="D")})

dim_date["date_key"]           = dim_date["date"].dt.strftime("%Y%m%d").astype(int)
dim_date["year"]               = dim_date["date"].dt.year
dim_date["quarter"]            = "Q" + dim_date["date"].dt.quarter.astype(str)
dim_date["month_number"]       = dim_date["date"].dt.month
dim_date["month_name"]         = dim_date["date"].dt.strftime("%b")
dim_date["year_month"]         = dim_date["date"].dt.strftime("%Y-%m")
dim_date["day_of_week_number"] = dim_date["date"].dt.dayofweek + 1
dim_date["day_name"]           = dim_date["date"].dt.strftime("%a")
dim_date["is_weekend"]         = dim_date["date"].dt.dayofweek >= 5

print("Rows:", len(dim_date))
dim_date.head()

Rows: 1096


,date,date_key,year,quarter,month_number,month_name,year_month,day_of_week_number,day_name,is_weekend
0,2016-01-01,20160101,2016,Q1,1,Jan,2016-01,5,Fri,False
1,2016-01-02,20160102,2016,Q1,1,Jan,2016-01,6,Sat,True
2,2016-01-03,20160103,2016,Q1,1,Jan,2016-01,7,Sun,True
3,2016-01-04,20160104,2016,Q1,1,Jan,2016-01,1,Mon,False
4,2016-01-05,20160105,2016,Q1,1,Jan,2016-01,2,Tue,False


In [63]:
print("Date range:", dim_date["date"].min().date(), "to", dim_date["date"].max().date())
print("Unique date keys:", dim_date["date_key"].nunique())
dim_date.tail(3)

Date range: 2016-01-01 to 2018-12-31
Unique date keys: 1096


,date,date_key,year,quarter,month_number,month_name,year_month,day_of_week_number,day_name,is_weekend
1093,2018-12-29,20181229,2018,Q4,12,Dec,2018-12,6,Sat,True
1094,2018-12-30,20181230,2018,Q4,12,Dec,2018-12,7,Sun,True
1095,2018-12-31,20181231,2018,Q4,12,Dec,2018-12,1,Mon,False


## Building the Order Items Fact Table

In [64]:
# Order details we need on every item row
order_info = orders_clean[["order_id", "customer_id", "order_status", "purchase_date"]]

# Link each customer_id to the customer_key of the real person
customer_map = customers[["customer_id", "customer_unique_id"]].merge(
    dim_customers[["customer_key", "customer_unique_id"]],
    on="customer_unique_id",
    how="left",
)[["customer_id", "customer_key"]]

# Start from the items and add each dimension key
fact_order_items = (
    items.merge(order_info, on="order_id", how="left")
         .merge(customer_map, on="customer_id", how="left")
         .merge(dim_products[["product_id", "product_key"]], on="product_id", how="left")
         .merge(dim_sellers[["seller_id", "seller_key"]], on="seller_id", how="left")
)

fact_order_items["date_key"] = fact_order_items["purchase_date"].dt.strftime("%Y%m%d").astype(int)
fact_order_items["item_total"] = fact_order_items["price"] + fact_order_items["freight_value"]
fact_order_items["is_revenue_order"] = ~fact_order_items["order_status"].isin(["canceled", "unavailable"])

fact_order_items = fact_order_items[[
    "order_id", "order_item_id", "customer_key", "product_key", "seller_key",
    "date_key", "order_status", "is_revenue_order",
    "price", "freight_value", "item_total",
]]

print("Rows:", len(fact_order_items))
fact_order_items.head()

Rows: 112650


,order_id,order_item_id,customer_key,product_key,seller_key,date_key,order_status,is_revenue_order,price,freight_value,item_total
0,00010242fe8c5a6d1ba2dd792cb16214,1,72412,25866,514,20170913,delivered,True,58.90,13.29,72.19
1,00018f77f2f0320c557190d7a144bdd3,1,71625,27231,472,20170426,delivered,True,239.90,19.93,259.83
2,000229ec398224ef6ca0657da4fc703e,1,49833,22625,1825,20180114,delivered,True,199.00,17.87,216.87
3,00024acbcdf0a6daa1e931b038114c75,1,4258,15404,2024,20180808,delivered,True,12.99,12.79,25.78
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,94830,8863,1598,20170204,delivered,True,199.90,18.14,218.04


In [65]:
print("Missing customer_key:", fact_order_items["customer_key"].isnull().sum())
print("Missing product_key:", fact_order_items["product_key"].isnull().sum())
print("Missing seller_key:", fact_order_items["seller_key"].isnull().sum())
print("Missing date_key:", fact_order_items["date_key"].isnull().sum())
print()

revenue_rows = fact_order_items[fact_order_items["is_revenue_order"]]
print("Revenue item rows:", len(revenue_rows))
print("Revenue orders:", revenue_rows["order_id"].nunique())
print("Product revenue:", round(revenue_rows["price"].sum(), 2))
print("Freight revenue:", round(revenue_rows["freight_value"].sum(), 2))

Missing customer_key: 0
Missing product_key: 0
Missing seller_key: 0
Missing date_key: 0

Revenue item rows: 112101
Revenue orders: 98199
Product revenue: 13494400.74
Freight revenue: 2241126.29


## Building the Orders Fact Table

In [66]:
order_items_summary = items.groupby("order_id").agg(
    items_count=("order_item_id", "count"),
    order_price=("price", "sum"),
    order_freight=("freight_value", "sum"),
).reset_index()

order_items_summary.head()

,order_id,items_count,order_price,order_freight
0,00010242fe8c5a6d1ba2dd792cb16214,1,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,199.90,18.14


In [67]:
fact_orders = (
    orders_clean.merge(customer_map, on="customer_id", how="left")
                .merge(order_items_summary, on="order_id", how="left")
                .merge(order_payments, on="order_id", how="left")
                .merge(order_reviews[["order_id", "review_score", "has_comment"]], on="order_id", how="left")
)

# Orders with no items have no price, freight or item count: set those to 0
for col in ["items_count", "order_price", "order_freight"]:
    fact_orders[col] = fact_orders[col].fillna(0)

fact_orders["date_key"] = fact_orders["purchase_date"].dt.strftime("%Y%m%d").astype(int)
fact_orders["is_revenue_order"] = ~fact_orders["order_status"].isin(["canceled", "unavailable"])
fact_orders["is_delivered"] = fact_orders["order_status"] == "delivered"

fact_orders = fact_orders[[
    "order_id", "customer_key", "date_key", "order_status",
    "is_revenue_order", "is_delivered",
    "order_purchase_timestamp", "order_approved_at",
    "order_delivered_carrier_date", "order_delivered_customer_date",
    "order_estimated_delivery_date",
    "delivery_days", "delay_days", "is_late",
    "items_count", "order_price", "order_freight",
    "total_payment_value", "payment_methods_used", "max_installments", "main_payment_type",
    "review_score", "has_comment",
]]

print("Rows:", len(fact_orders))
fact_orders.head()

Rows: 99441


,order_id,customer_key,date_key,order_status,is_revenue_order,is_delivered,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,...,is_late,items_count,order_price,order_freight,total_payment_value,payment_methods_used,max_installments,main_payment_type,review_score,has_comment
0,e481f51cbdc54678b7cc49136f2d6af7,69809,20171002,delivered,True,True,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,...,False,1.0,29.99,8.72,38.71,2.0,1.0,voucher,4.0,True
1,53cdb2fc8bc7dce0b6741e2150273451,8157,20180724,delivered,True,True,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,...,False,1.0,118.70,22.76,141.46,1.0,1.0,boleto,4.0,True
2,47770eb9100c2d0c44946d9cf07ec65d,4277,20180808,delivered,True,True,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,...,False,1.0,159.90,19.22,179.12,1.0,3.0,credit_card,5.0,False
3,949d5b44dbf5de918fe9c16f97b45f8a,62672,20171118,delivered,True,True,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,...,False,1.0,45.00,27.20,72.20,1.0,1.0,credit_card,5.0,True
4,ad21c59c0840e6cb83a9ceb5573f8159,43001,20180213,delivered,True,True,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,...,False,1.0,19.90,8.72,28.62,1.0,1.0,credit_card,5.0,False


In [68]:
print("Unique orders:", fact_orders["order_id"].nunique())
print("Missing customer_key:", fact_orders["customer_key"].isnull().sum())
print()

revenue_orders = fact_orders[fact_orders["is_revenue_order"]]
print("Revenue orders:", len(revenue_orders))
print("Product revenue:", round(revenue_orders["order_price"].sum(), 2))
print("Average order value (price only):", round(revenue_orders["order_price"].sum() / len(revenue_orders), 2))
print()
print("Average review score:", round(fact_orders["review_score"].mean(), 2))
print("Orders with no review:", fact_orders["review_score"].isnull().sum())
print("Orders with no payment record:", fact_orders["total_payment_value"].isnull().sum())

Unique orders: 99441
Missing customer_key: 0

Revenue orders: 98207
Product revenue: 13494400.74
Average order value (price only): 137.41

Average review score: 4.09
Orders with no review: 768
Orders with no payment record: 1


fact_orders["items_count"] = fact_orders["items_count"].astype(int)

for col in ["payment_methods_used", "max_installments", "review_score"]:
    fact_orders[col] = fact_orders[col].astype("Int64")

fact_orders[["items_count", "payment_methods_used", "max_installments", "review_score"]].head()

## Fixing the Revenue Order Rule So Orders Without Items Don't Count

In [69]:
fact_orders["is_revenue_order"] = (
    fact_orders["is_revenue_order"] & (fact_orders["items_count"] > 0)
)

revenue_orders = fact_orders[fact_orders["is_revenue_order"]]
print("Revenue orders:", len(revenue_orders))
print("Product revenue:", round(revenue_orders["order_price"].sum(), 2))
print("Average order value (price only):", round(revenue_orders["order_price"].sum() / len(revenue_orders), 2))

Revenue orders: 98199
Product revenue: 13494400.74
Average order value (price only): 137.42


## Saving the Clean Warehouse Tables

In [70]:
PROCESSED_DIR = "../data/processed"

warehouse_tables = {
    "dim_customers": dim_customers,
    "dim_products": dim_products,
    "dim_sellers": dim_sellers,
    "dim_date": dim_date,
    "fact_order_items": fact_order_items,
    "fact_orders": fact_orders,
}

for name, table in warehouse_tables.items():
    table.to_csv(f"{PROCESSED_DIR}/{name}.csv", index=False)
    print(f"{name}: {table.shape[0]:,} rows, {table.shape[1]} columns")

dim_customers: 96,096 rows, 7 columns
dim_products: 32,951 rows, 11 columns
dim_sellers: 3,095 rows, 7 columns
dim_date: 1,096 rows, 10 columns
fact_order_items: 112,650 rows, 11 columns
fact_orders: 99,441 rows, 23 columns


In [71]:
for file in sorted(os.listdir(PROCESSED_DIR)):
    size_mb = os.path.getsize(f"{PROCESSED_DIR}/{file}") / (1024 * 1024)
    print(f"{file}: {size_mb:.1f} MB")

dim_customers.csv: 9.0 MB
dim_date.csv: 0.1 MB
dim_products.csv: 3.6 MB
dim_sellers.csv: 0.3 MB
fact_order_items.csv: 10.3 MB
fact_orders.csv: 21.4 MB


## Connecting the Notebook to PostgreSQL

In [72]:
%pip install sqlalchemy psycopg2-binary

Note: you may need to restart the kernel to use updated packages.


In [73]:
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL
import getpass

db_password = getpass.getpass("PostgreSQL password: ")

engine = create_engine(URL.create(
    "postgresql+psycopg2",
    username="postgres",
    password=db_password,
    host="localhost",
    port=5432,
    database="ecommerce_analytics_dw",
))

with engine.connect() as conn:
    print("Connected to:", conn.execute(text("SELECT current_database()")).scalar())

PostgreSQL password:  ········


Connected to: ecommerce_analytics_dw


## Loading the Clean Tables into the PostgreSQL Warehouse

In [74]:
# Empty the warehouse tables first, so this cell can be re-run safely
with engine.begin() as conn:
    conn.execute(text("""
        TRUNCATE dw.fact_order_items, dw.fact_orders,
                 dw.dim_customers, dw.dim_products, dw.dim_sellers, dw.dim_date
    """))

# Dimensions first, then the fact tables that refer to them
load_order = [
    ("dim_customers", dim_customers),
    ("dim_products", dim_products),
    ("dim_sellers", dim_sellers),
    ("dim_date", dim_date),
    ("fact_orders", fact_orders),
    ("fact_order_items", fact_order_items),
]

for name, table in load_order:
    table.to_sql(name, engine, schema="dw", if_exists="append",
                 index=False, method="multi", chunksize=2000)
    print(f"Loaded {name}: {len(table):,} rows")

Loaded dim_customers: 96,096 rows
Loaded dim_products: 32,951 rows
Loaded dim_sellers: 3,095 rows
Loaded dim_date: 1,096 rows
Loaded fact_orders: 99,441 rows
Loaded fact_order_items: 112,650 rows


In [75]:
checks = {
    "dim_customers": "SELECT COUNT(*) FROM dw.dim_customers",
    "dim_products": "SELECT COUNT(*) FROM dw.dim_products",
    "dim_sellers": "SELECT COUNT(*) FROM dw.dim_sellers",
    "dim_date": "SELECT COUNT(*) FROM dw.dim_date",
    "fact_orders": "SELECT COUNT(*) FROM dw.fact_orders",
    "fact_order_items": "SELECT COUNT(*) FROM dw.fact_order_items",
    "revenue orders": "SELECT COUNT(*) FROM dw.fact_orders WHERE is_revenue_order",
    "product revenue": "SELECT ROUND(SUM(price), 2) FROM dw.fact_order_items WHERE is_revenue_order",
}

with engine.connect() as conn:
    for label, query in checks.items():
        print(f"{label}: {conn.execute(text(query)).scalar():,}")

dim_customers: 96,096
dim_products: 32,951
dim_sellers: 3,095
dim_date: 1,096
fact_orders: 99,441
fact_order_items: 112,650
revenue orders: 98,199
product revenue: 13,494,400.74
